In [1]:
import os
import csv
import numpy as np
import pandas as pd

COL_TX_ID = 0
COL_RX_ID = 1
COL_RX_TIMESTAMP = 2
COL_RSSI = 3
COL_SCEN_ID = 4
COLUMNS = ["sender_id", "receiver_id", "timestamp", "rssi", "scenario_id"]

FIRST_ANCHOR_ID = 29 # From this ID on, nodes are static

WARMUP_TIME = 180
MIN_LEN = 10
MAX_PACKETS = 150

cur_dir = os.getcwd()

csv_folder = f"{cur_dir}/rssi_traces"
csv_output = "urban_rpl_mobility_sequences.csv"
output_path = f'{csv_folder}/{csv_output}'

csv_files = [
    os.path.join(csv_folder, f)
    for f in os.listdir(csv_folder)
    if "run" in f
]

dfs = []
for i, file in enumerate(csv_files):
    print(f"Loading {file}")
    temp = pd.read_csv(file)
    temp["scenario_id"] = i
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(by=['scenario_id', 'sender_id', 'receiver_id', 'timestamp']).reset_index(drop=True)

Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run10.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run01.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run05.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run09.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run07.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run08.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run02.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run06.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run04.csv
Loading /home/joaog/Documentos/Jupyter/Dataset Repo/rssi_traces/manhattan_run03.csv


In [2]:
def get_sequences_from_df(df, col1, col2):
    # Identify group boundaries: whenever either col1 or col2 changes
    groups = ((df[col1] != df[col1].shift(1)) | (df[col2] != df[col2].shift(1))).cumsum()

    # Split DataFrame into sub-DataFrames, then convert each to NumPy arrays
    numpy_sequences = [group[COLUMNS].to_numpy() for _, group in df.groupby(groups)]

    return numpy_sequences

sequences = get_sequences_from_df(df, 'receiver_id', 'sender_id')

# Pre-filtering, remove anchor-anchor links
sequences = [seq for seq in sequences if not (seq[0, COL_TX_ID] >= FIRST_ANCHOR_ID and seq[0, COL_RX_ID] >= FIRST_ANCHOR_ID)]

lengths = [len(seq) for seq in sequences]

lengths = np.array(lengths)

print("Min:", lengths.min())
print("Max:", lengths.max())
print("Mean:", lengths.mean())
print("Median:", np.median(lengths))
print("Std:", lengths.std())

Min: 1
Max: 411
Mean: 113.96317922648898
Median: 108.0
Std: 48.96017357142239


In [3]:
# Construct observation vector dataset
# Large inter packet gaps are not cut out, but left in the previous sequence
SIM_START = 0 #In seconds
SIM_END = 3600 #In seconds
TX_INTERVAL = 1 #In seconds
WINDOW = 1 #In seconds
WARM_START_THRESH = 10 #In seconds, must be lower than LINK_BREAK_THRESH
LINK_BREAK_THRESH = 20

LINK_BREAK_UPDATES_EP_ID = 1
USE_WARM_START = 1

if os.path.isfile(output_path):
    os.remove(output_path)
with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["scenario_id", "ep_id", "sender_id", "receiver_id", "packet_count", "timestamp", "time_since_last_rx", "rx_delta_t", "rssi"])
    ep_no = 0
    for seq in sequences:
        #Figure out initial observation timestamp
        k = 0
        while SIM_START + k*WINDOW < seq[0, COL_RX_TIMESTAMP]:
            k += 1
        i = k
        timestamp = SIM_START + i*WINDOW
        j = j_prev = 0
        l = 0
        while timestamp < seq[-1, COL_RX_TIMESTAMP] + WINDOW and timestamp <= SIM_END:
            while j < len(seq) and timestamp >= seq[j, COL_RX_TIMESTAMP]:
                rssi = seq[j, COL_RSSI]
                if l == 0:
                    delta_t = TX_INTERVAL #Initial estimate should be the typical transmission interval
                elif l == 1:
                    delta_t = seq[j, COL_RX_TIMESTAMP] - seq[j-1, COL_RX_TIMESTAMP]
                else:
                    delta_t = seq[j, COL_RX_TIMESTAMP] - seq[j-1, COL_RX_TIMESTAMP]
                j += 1
                l += 1
            time_since_rx = timestamp - seq[j-1, COL_RX_TIMESTAMP]
            writer.writerow([seq[0, COL_SCEN_ID], ep_no, seq[0, COL_TX_ID], seq[0, COL_RX_ID], l, timestamp, time_since_rx, delta_t, rssi])
            j_prev = j
            i += 1
            if LINK_BREAK_UPDATES_EP_ID:
                gap = time_since_rx
                if gap >= LINK_BREAK_THRESH or (not USE_WARM_START and gap >= WARM_START_THRESH):
                    if j >= len(seq):
                        break
                    ep_no += 1
                    l = 0 #Ensures reinitialization in next iteration via "if l == 0"
                    k = 0
                    while SIM_START + k*WINDOW < seq[j, COL_RX_TIMESTAMP]:
                        k += 1
                    i = k
            timestamp = SIM_START + i*WINDOW
        ep_no += 1

processed_output_path = f'{csv_folder}/{csv_output.split(".")[0]}_obsv_minlen={MIN_LEN}_linkbreak={LINK_BREAK_THRESH}s.csv'

In [4]:
file_path = output_path

processed_df = pd.read_csv(file_path)

# Initial filter: keep only entries with timestamp >= 180 s
processed_df = processed_df[processed_df["timestamp"] >= WARMUP_TIME]

# Offset timestamps so they start at 0
processed_df["timestamp"] = processed_df["timestamp"] - WARMUP_TIME

filtered_df = processed_df.copy()

new_rows = []
new_ep = 0

for _, group in filtered_df.groupby("ep_id"):
    group = group.sort_values("timestamp").copy().reset_index(drop=True)

    is_new_packet = (group["packet_count"].diff() > 0)
    is_new_packet.iloc[0] = True

    packet_counter = 0
    start_idx = 0
    cut_pending = False

    for i in range(len(group)):
        if is_new_packet.iloc[i]:
            packet_counter += 1
        
            if cut_pending:
                # Cut here
                end_idx = i
        
                chunk = group.iloc[start_idx:end_idx].copy()
                chunk["ep_id"] = new_ep
                new_rows.append(chunk)
                new_ep += 1
        
                # Restart
                start_idx = i
                packet_counter = 1
                cut_pending = False
        
            elif packet_counter >= MAX_PACKETS:
                cut_pending = True

    # Adds final chunk
    if start_idx < len(group):
        chunk = group.iloc[start_idx:].copy()
        chunk["ep_id"] = new_ep
        new_rows.append(chunk)
        new_ep += 1
filtered_df = pd.concat(new_rows, ignore_index=True)

seq_packets = filtered_df.groupby("ep_id")["packet_count"].nunique()
valid_eps = seq_packets[seq_packets >= MIN_LEN].index
filtered_df = filtered_df[filtered_df["ep_id"].isin(valid_eps)]
ep_ids = filtered_df["ep_id"].unique()
np.random.shuffle(ep_ids)
filtered_df = (filtered_df.set_index("ep_id").loc[ep_ids].reset_index())
filtered_df.to_csv(processed_output_path, index=False)
print(f"Sequences after filtering: {filtered_df['ep_id'].nunique()}")
print(f"Observations after filtering: {len(filtered_df)}")

print("Packet count information, post filtering: \n")
filtered_df.groupby("ep_id")["packet_count"].nunique().describe(
    percentiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
)

Sequences after filtering: 55505
Observations after filtering: 5167596
Packet count information, post filtering: 



count    55505.000000
mean        18.112620
std          9.672747
min         10.000000
10%         10.000000
20%         11.000000
30%         12.000000
40%         14.000000
50%         15.000000
60%         17.000000
70%         19.000000
80%         23.000000
90%         29.000000
max        150.000000
Name: packet_count, dtype: float64